# 13. Návrhové vzory - creational design patterns, structural design patterns, behavioral patterns

### Co jsou návrhové vzory (Design Patterns)
* Zavedená a ověřená architektonická řešení nejčastějších programátorských problémů
* Ulehčují čitelnost kódu, jeho strukturu a spolehlivost

### Creational design patterns (Vytvářecí vzory)
* Zajišťují proces vytváření objektů, aby byl bezpečný, efektivní a skrytý před uživatelem
* Singleton - V celém programu vždy jedna instance dané třídy, typicky pro připojení k databázi
* Factory method - Tvorba objektu probíhá ve vyhrazené funkci, tvořící proces lze poté obalit do bloku `try/catch`, vyřešit chybové stavy a při selhání místo pádu aplikace vrátit objekt `None` nebo pokus zopakovat
* Lazy initialization - Odkládá reálné vytvoření objektu v paměti do doby, kdy je poprvé provedena operace zápisu nebo čtení

### Structural design patterns
* Uspořádání tříd do logických celků a skrytí složitosti vnitřního systému
* Facade - K zakrytí složitosti celého podsystému (např. práce s databází) za jednoduché rozhraní, fasáda může nabídnout CRUD metody, zatímco SQL připojení, inserty a updaty se řeší bez vědomí hlavního programu uvnitř
* DAO (Data Access Object) / Table Gateway - Pro každou tabulku v databázi existuje v programu právě jedna konkrétní třída, řešící manipulaci s ní
* Active Record / Row Gateway - Pracuje přímo s databázovými záznamy (řádky), existuje jedna třída pro každý řádek a objekt sám odpovídá za to, že se udělá UPDATE nebo INSERT do databáze

### Behavioral patterns
* Jakým způsobem spolu objekty komunikují a jak dynamicky mění své chování
* State pattern a Strategy pattern
    * Třída mění své chování podle vnitřního stavu nebo vložené strategie
    * Využívá se například při programování TCP serveru
    * Pokud server řeší zjištění proměnných pro výpočet Ohmova zákona, tak výchozí objekt StateKnowNothing přijme napětí U, na základě toho se smaže a dynamicky se nahradí stavem StateKnowU, který při příštím vstupu očekává zadání I nebo R, pokud má dost informací k výpočtu, výsledek vrátí (stav StateKnowRandU)

**1. CREATIONAL PATTERNS: Singleton a Factory Method**

In [ ]:
class DatabaseConnection:
    _instance = None # Třídní proměnná udržující jedinou instanci (Singleton)

    def __init__(self):
        # Obrana před vytvořením instance na přímo
        raise Exception("Toto je Singleton, použijte tovární metodu get_instance()!")

    @classmethod
    def get_instance(cls):
        # Lazy Initialization a Factory Method
        if cls._instance is None:
            try:
                print("[FACTORY] Vytvářím unikátní spojení do databáze...")
                cls._instance = object.__new__(cls)
                cls._instance.is_connected = True
            except Exception as e:
                print(f"[FACTORY] Chyba: {e}")
                return None # Náhrada padající výjimky
        return cls._instance

**2. STRUCTURAL PATTERNS: Facade pro CRUD**

In [ ]:
class UzivatelFacade:
    """
    Fasáda skrývá SQL operace a Singleton volání.
    Hlavní program díky ní volá jen jednoduché rozhraní.
    """
    def __init__(self):
        self.db = DatabaseConnection.get_instance()

    def create_user(self, username, favorite_number, favorite_color):
        print(f"[FACADE C-R-U-D] Ukládám do DB: {username}, Barva: {favorite_color}")

**3. BEHAVIORAL PATTERNS: State Pattern**

In [ ]:
class StateKnowNothing:
    def prijem_zpravy(self, kontext, klic, hodnota):
        if klic == "U":
            print(f"[STATE] Nyní znám napětí U = {hodnota}. Měním chování na StateKnowU.")
            kontext.stav = StateKnowU() # Nahrazení sebe sama novým chováním

class StateKnowU:
    def prijem_zpravy(self, kontext, klic, hodnota):
        if klic == "I":
            print(f"[STATE] Znám U i I. Spouštím výpočet odporu R = U / I.")
            # Následně by přešel do dalšího stavu (např. StateKnowUandI)

class TcpServerContext:
    def __init__(self):
        # Objekt drží svůj aktuální stav
        self.stav = StateKnowNothing()

    def tcp_zprava(self, klic, hodnota):
        # Zpracování deleguje vždy na chování aktuálního vloženého stavu
        self.stav.prijem_zpravy(self, klic, hodnota)

**TESTOVÁNÍ V JUPYTERU**

In [ ]:
print(">>> STRUKTURÁLNÍ A VYTVÁŘEJÍCÍ VZORY")
aplikacni_fasada = UzivatelFacade()
aplikacni_fasada.create_user("Vilém", 4, "růžová")

print("\n>>> BEHAVIORÁLNÍ VZORY")
server = TcpServerContext()
# Chování v prvním stavu
server.tcp_zprava("U", 230)
# Následující chování už ovládá jiný objekt stavu
server.tcp_zprava("I", 10)